# IncStrad – Microdati → DataFrame

Questo notebook trasforma i microdati ISTAT sugli **incidenti stradali** (formato fixed-width) in un DataFrame pandas.

**Struttura attesa delle cartelle:**
```
dati/
  INCSTRAD_<ANNO>_IT/
    MICRODATI/
      IncStrad_Microdati_Anno_<ANNO>.txt
    METADATI/
      IncStrad_Tracciato_Anno <ANNO>.html
      Classificazioni/
        IncStrad_Classificazione_Anno <ANNO>_var<N>.html
```

**Passaggi:**
1. Parsing del tracciato HTML → nome campo, lunghezza, tipo, link decodifica
2. Calcolo delle posizioni di inizio (somma cumulativa delle lunghezze)
3. Parsing delle tabelle di decodifica
4. Lettura dei file fixed-width con `pd.read_fwf`
5. Conversione tipi e applicazione decodifiche
6. Esportazione del DataFrame combinato

In [ ]:
!pip install -q beautifulsoup4 lxml pandas pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Diagnostica: trova il percorso corretto su Google Drive

Se non sai il percorso esatto dei dati su Drive, esegui la cella qui sotto:
elencherà la struttura di Drive e **suggerirà il valore corretto di `BASE_DIR`**.

In [ ]:
# ================================================================
# DIAGNOSTICA — esegui questa cella per trovare BASE_DIR corretto
# poi aggiorna BASE_DIR nella cella CONFIGURAZIONE qui sotto
# ================================================================
import glob as _glob, os as _os

print('=== 1. Drive montato? ===')
drive_ok = _os.path.exists('/content/drive/MyDrive')
print('OK' if drive_ok else 'NO — decommenta drive.mount() nella cella sopra e riesegui')
print()

if drive_ok:
    print('=== 2. Primo livello di MyDrive ===')
    for item in sorted(_os.listdir('/content/drive/MyDrive'))[:30]:
        print(' ', item)
    print()

    print('=== 3. Ricerca file microdati .txt (può richiedere qualche secondo) ===')
    hits = _glob.glob('/content/drive/MyDrive/**/*.txt', recursive=True)
    microdati_hits = [h for h in hits if 'MICRODATI' in h]

    if microdati_hits:
        for h in microdati_hits[:10]:
            print(' ', h)
        sample = microdati_hits[0]
        # Risale 3 livelli: .../dati/INCSTRAD_ANNO_IT/MICRODATI/file.txt → dati/
        candidate = _os.path.normpath(
            _os.path.join(_os.path.dirname(sample), '..', '..', '..')
        )
        print(f'\n→ BASE_DIR suggerito: "{candidate}"')
        print('  Copia questo valore nella variabile BASE_DIR della cella CONFIGURAZIONE.')
    elif hits:
        print('  Trovati .txt ma nessuno in cartella MICRODATI:')
        for h in hits[:5]:
            print(' ', h)
    else:
        print('  Nessun file .txt trovato su Drive.')
        print('  Verifica di aver copiato la cartella dati/INCSTRAD_*/ su Google Drive.')

In [ ]:
# ================================================================
# CONFIGURAZIONE — modifica solo questi parametri
# ================================================================
import os

# Cartella che contiene le sottocartelle INCSTRAD_<ANNO>_IT/
BASE_DIR = '/content/drive/MyDrive/TesiMagistrale/FontiUsate/dati'

# Encoding dei file di microdati (tab-separated .txt) — ISTAT usa latin-1
DATA_ENCODING = 'latin-1'

# Se True, aggiunge colonne '<campo>_label' con le etichette decodificate
APPLY_DECODING = True

# Se True, aggiunge la colonna 'anno' ricavata dal path del file
ADD_YEAR_COLUMN = True

# Colonne da conservare nel DataFrame finale.
# Tutto il resto viene scartato subito dopo la lettura per risparmiare memoria.
COLS_NEEDED = [
    'provincia',
    'comune',
    'mese',
    'giorno_settimana',
    'ora',
    'localizzazione_incidente',
    'natura_incidente',
    'tipo_veicolo_a',
    'tipo_veicoli__b_',
    'tipo_veicolo__c_',
    'morti_entro_24_ore',
    'morti_entro_30_giorni',
    'feriti',
]

print(f'BASE_DIR : {BASE_DIR}')
print(f'Esiste?  : {os.path.exists(BASE_DIR)}')

In [ ]:
import os
import re
import glob
import warnings
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup

warnings.filterwarnings('ignore')
print('Librerie importate.')

## 1. Parsing del tracciato HTML

Il tracciato ha 13 colonne. Le posizioni di inizio **non sono nel file**: vengono calcolate come somma cumulativa delle lunghezze.

| Indice | Colonna         | Uso |
|--------|-----------------|-----|
| 1      | Lunghezza       | lunghezza del campo |
| 2      | Nome Campo      | nome colonna DataFrame |
| 3      | TipoVariabile   | link `Categorica` → tabella di decodifica |
| 8      | Formato Campo   | `A`=stringa, `N`=numerico |
| 9      | Num. decimali   | numero decimali (per i float) |
| 10     | Sep. decimali   | separatore decimale |

In [ ]:
def _open_html(html_path):
    """Apre un file HTML provando prima utf-8, poi latin-1."""
    for enc in ('utf-8-sig', 'utf-8', 'latin-1'):
        try:
            with open(html_path, encoding=enc) as f:
                content = f.read()
            return content
        except UnicodeDecodeError:
            continue
    # fallback con sostituzione
    with open(html_path, encoding='latin-1', errors='replace') as f:
        return f.read()


def parse_tracciato(html_path, encoding=None):
    """
    Legge il tracciato HTML e restituisce:
      - fields: lista di dict {nome, start, length, formato, n_dec, sep_dec, descrizione}
      - decode_links: dict {nome_campo: path_assoluto_html_decodifica}

    La posizione iniziale (start, 0-based) è calcolata come somma cumulativa
    delle lunghezze dei campi precedenti.
    """
    soup = BeautifulSoup(_open_html(html_path), 'lxml')

    table = soup.find('table')
    if table is None:
        raise ValueError(f'Nessuna tabella trovata in {html_path}')

    rows = table.find_all('tr')
    # Riga 0: titolo (colspan=13)  → saltare
    # Riga 1: intestazione (bgcolor=orange, usa <td> non <th>) → saltare
    # Righe 2+: dati

    decode_dir = os.path.dirname(html_path)
    fields = []
    decode_links = {}
    cursor = 0  # posizione corrente (0-based)

    for row in rows[2:]:
        cells = row.find_all('td')
        if len(cells) < 9:
            continue

        length_raw = cells[1].get_text(strip=True)
        nome = cells[2].get_text(strip=True)
        formato = cells[8].get_text(strip=True).upper()  # A o N
        n_dec_raw = cells[9].get_text(strip=True) if len(cells) > 9 else ''
        sep_dec = cells[10].get_text(strip=True) if len(cells) > 10 else ''
        descrizione = cells[7].get_text(strip=True) if len(cells) > 7 else ''

        # Salta righe non valide (testa tabelle annidate, celle vuote)
        if not nome or not re.match(r'^\d+$', length_raw):
            continue

        length = int(length_raw)
        n_dec = int(n_dec_raw) if re.match(r'^\d+$', n_dec_raw) else 0

        field = {
            'nome': nome,
            'start': cursor,
            'length': length,
            'formato': formato,
            'n_dec': n_dec,
            'sep_dec': sep_dec,
            'descrizione': descrizione,
        }
        fields.append(field)

        # Cerca link di decodifica in cells[3] (TipoVariabile)
        link_tag = cells[3].find('a', href=True)
        if link_tag:
            href = link_tag['href'].replace('./', '', 1)  # rimuove ./ iniziale
            decode_links[nome] = os.path.join(decode_dir, href)

        cursor += length

    print(f'Campi trovati       : {len(fields)}')
    print(f'Larghezza record    : {cursor} caratteri')
    print(f'Campi con decodifica: {len(decode_links)}')
    return fields, decode_links

### Test: verifica il tracciato del 2010

In [ ]:
# Trova il primo tracciato disponibile per verifica
tracciati = sorted(glob.glob(
    os.path.join(BASE_DIR, '*', 'METADATI', 'IncStrad_Tracciato_Anno*.html')
))
print(f'Tracciati trovati: {len(tracciati)}')
for t in tracciati:
    print(' ', t)

if tracciati:
    sample_fields, sample_decode_links = parse_tracciato(tracciati[0])
    df_tracciato = pd.DataFrame(sample_fields)
    print()
    display(df_tracciato.head(10))
    # Verifica: start del campo 1 = 0, campo 2 = 2, campo 3 = 4, ...
    print('\nVerifica posizioni inizio:')
    print(df_tracciato[['nome', 'start', 'length', 'formato', 'n_dec']].to_string(index=False))

## 2. Parsing delle tabelle di decodifica

Ogni HTML di classificazione ha una tabella con intestazione `bgcolor=orange` (usa `<td>`, non `<th>`).
- Colonna 0: codice
- Colonna 1: etichetta

In [ ]:
def parse_decode_table(html_path, encoding=None):
    """
    Legge un HTML di classificazione e restituisce un dict di decodifica.

    Gestisce due formati:
    - 2 colonne: {codice: etichetta}
    - 3 colonne (es. comuni): {provincia_code + comune_code: nome_comune}
      dove col0 = codice comune, col1 = nome, col2 = codice provincia
      → chiave = col2 + col0  (es. '001' + '026' = '001026')

    La prima riga (intestazione orange) viene saltata.
    """
    if not os.path.exists(html_path):
        return {}

    soup = BeautifulSoup(_open_html(html_path), 'lxml')
    table = soup.find('table')
    if table is None:
        return {}

    mapping = {}
    rows = table.find_all('tr')
    for row in rows[1:]:  # salta riga 0 (intestazione orange)
        cells = row.find_all('td')
        if len(cells) >= 3:
            c0 = cells[0].get_text(strip=True)  # codice comune
            c1 = cells[1].get_text(strip=True)  # nome comune
            c2 = cells[2].get_text(strip=True)  # codice provincia
            if c0 and c1 and c2:
                mapping[c2 + c0] = c1           # '001026' → 'Bobbio Pellice'
        elif len(cells) == 2:
            c0 = cells[0].get_text(strip=True)
            c1 = cells[1].get_text(strip=True)
            if c0 and c1:
                mapping[c0] = c1

    return mapping

## 3. Lettura del file fixed-width e conversione tipi

In [ ]:
def read_microdata(filepath, fields, encoding=DATA_ENCODING):
    """Legge un file di microdati separato da TAB."""
    names = [f['nome'] for f in fields]
    df = pd.read_csv(
        filepath,
        sep='\t',
        names=names,
        header=None,
        encoding=encoding,
        dtype=str,
        on_bad_lines='warn',
    )
    return df


def subset_columns(df):
    """
    Rinomina 'anno' → 'anno_ult2' (evita conflitti) e mantiene
    solo le colonne in COLS_NEEDED. Libera subito la memoria delle altre.
    """
    if 'anno' in df.columns:
        df = df.rename(columns={'anno': 'anno_ult2'})
    cols_present = [c for c in COLS_NEEDED if c in df.columns]
    return df[cols_present].copy()


# Campi che non devono mai essere vuoti e devono essere tutti decodificabili
QC_MANDATORY = [
    'localizzazione_incidente',
    'natura_incidente',
    'tipo_veicolo_a',
    'tipo_veicoli__b_',
    'tipo_veicolo__c_',
]


def run_quality_checks(df, decode_tables, anno):
    """
    Esegue i controlli di qualità su TUTTE le righe (prima del filtro).
    Opera sulle stringhe grezze per essere fedele al dato originale.
    """
    n = len(df)
    print(f'  --- QC [{anno}] su {n:,} righe ---')

    # 1. Righe con morti e feriti tutti a 0 o null
    num_fields = ['morti_entro_24_ore', 'morti_entro_30_giorni', 'feriti']
    zero_mask = pd.Series(True, index=df.index)
    for col in num_fields:
        if col in df.columns:
            zero_mask &= pd.to_numeric(df[col], errors='coerce').fillna(0) == 0
    n_zero = zero_mask.sum()
    flag = '⚠' if n_zero > 0 else '✔'
    print(f'  {flag} morti e feriti tutti a 0: {n_zero:,} ({n_zero/n*100:.1f}%)')

    # 2. Campi obbligatori: vuoti + copertura decodifica
    for campo in QC_MANDATORY:
        if campo not in df.columns:
            print(f'  ⚠ {campo}: colonna assente')
            continue

        val = df[campo].astype(str).str.strip()
        n_vuoti = (df[campo].isna() | (val == '') | (val == 'nan')).sum()

        if n_vuoti > 0:
            print(f'  ⚠ {campo}: {n_vuoti:,} vuoti ({n_vuoti/n*100:.1f}%)')
        else:
            print(f'  ✔ {campo}: nessun valore vuoto')

        if campo in decode_tables:
            valid = val[~(df[campo].isna() | (val == '') | (val == 'nan'))]
            not_decoded = ~valid.isin(decode_tables[campo].keys())
            n_nd = not_decoded.sum()
            if n_nd > 0:
                top = valid[not_decoded].value_counts().head(3).to_dict()
                print(f'  ⚠ {campo}: {n_nd:,} valori non decodificabili → {top}')
            else:
                print(f'  ✔ {campo}: tutti i valori decodificabili')


def filter_rows(df):
    """
    Filtra le righe mantenendo solo gli incidenti in ambito urbano/extraurbano:
      - localizzazione_incidente in [0, 1, 2, 3]
    I sotto-filtri per natura/tipo veicolo vengono applicati a livello di pivot.
    """
    if 'localizzazione_incidente' in df.columns:
        loc_mask = df['localizzazione_incidente'].str.strip().isin(['0','1','2','3'])
    else:
        loc_mask = pd.Series(True, index=df.index)
    return df[loc_mask].copy()


def convert_types(df, fields):
    """
    Converte i tipi delle colonne presenti:
      - formato A  → stringa (strip spazi)
      - formato N, n_dec == 0 → Int64
      - formato N, n_dec >  0 → float
    """
    field_map = {f['nome']: f for f in fields}
    for col in df.columns:
        f = field_map.get(col)
        if f is None:
            continue
        if f['formato'] == 'N':
            if f['n_dec'] > 0:
                sep = f['sep_dec'] if f['sep_dec'] else '.'
                if sep != '.':
                    df[col] = df[col].str.replace(sep, '.', regex=False)
                df[col] = pd.to_numeric(df[col].str.strip(), errors='coerce')
            else:
                df[col] = pd.to_numeric(df[col].str.strip(), errors='coerce').astype('Int64')
        else:
            df[col] = df[col].str.strip()
    return df


def apply_decodings(df, decode_tables):
    """
    Aggiunge '<campo>_label' per ogni campo con tabella di decodifica.

    Se la chiave del dict è più lunga del valore nel campo (es. '001026' vs '026'),
    significa che è una chiave combinata provincia+comune: costruisce la chiave
    come zfill(3) di provincia + zfill(3) di comune.
    """
    for campo, mapping in decode_tables.items():
        if campo not in df.columns or not mapping:
            continue

        first_key = next(iter(mapping))
        sample_val = df[campo].astype(str).str.strip().dropna().iloc[0] if len(df) > 0 else ''

        if len(first_key) > len(sample_val) + 1 and 'provincia' in df.columns:
            # chiave combinata: provincia + comune, zero-padded a 3 cifre ciascuno
            keys = (df['provincia'].astype(str).str.strip().str.zfill(3)
                    + df[campo].astype(str).str.strip().str.zfill(3))
            df[campo + '_label'] = keys.map(mapping)
        else:
            df[campo + '_label'] = df[campo].astype(str).str.strip().map(mapping)
    return df


def extract_year(path):
    """Estrae il primo anno a 4 cifre (1900-2099) dal path."""
    m = re.search(r'((?:19|20)\d{2})', str(path))
    return int(m.group(1)) if m else None

## 4. Elaborazione di tutti gli anni

In [ ]:
# Trova tutti i file di microdati
data_files = sorted(glob.glob(
    os.path.join(BASE_DIR, '*', 'MICRODATI', '*.txt')
))
print(f'File di microdati trovati: {len(data_files)}')
for p in data_files:
    print(' ', p)

In [ ]:
all_dfs = []

for filepath in data_files:
    anno = extract_year(filepath)
    print(f'\n[{anno}] {os.path.basename(filepath)}')

    metadati_dir = os.path.normpath(
        os.path.join(os.path.dirname(filepath), '..', 'METADATI')
    )
    try:
        tracciati = [
            os.path.join(metadati_dir, f)
            for f in os.listdir(metadati_dir)
            if 'incstrad' in f.lower()
            and 'tracciato' in f.lower()
            and f.lower().endswith('.html')
        ]
    except FileNotFoundError:
        tracciati = []

    if not tracciati:
        print(f'  ATTENZIONE: nessun tracciato trovato in {metadati_dir}')
        continue

    tracciato_path = tracciati[0]
    print(f'  Tracciato: {os.path.basename(tracciato_path)}')

    try:
        # 1. Parsing tracciato
        fields, decode_links = parse_tracciato(tracciato_path)

        # 2. Caricamento tabelle di decodifica (solo per colonne che useremo)
        decode_tables = {}
        if APPLY_DECODING:
            for campo, html_path in decode_links.items():
                if campo in COLS_NEEDED:
                    tbl = parse_decode_table(html_path)
                    if tbl:
                        decode_tables[campo] = tbl

        # 3. Lettura e riduzione colonne
        df = read_microdata(filepath, fields)
        df = subset_columns(df)

        # 4. QC su TUTTE le righe (prima del filtro)
        run_quality_checks(df, decode_tables, anno)

        # 5. Filtro righe
        df = filter_rows(df)
        print(f'  Righe dopo filtro: {len(df):,}')

        # 6. Decodifiche PRIMA della conversione tipi:
        #    i codici sono ancora stringhe grezze (es. '038'), identiche alle chiavi HTML
        if APPLY_DECODING:
            df = apply_decodings(df, decode_tables)

        # 7. Conversione tipi (dopo le decodifiche per evitare disallineamenti nei codici)
        df = convert_types(df, fields)

        # 8. Colonna anno (4 cifre)
        if ADD_YEAR_COLUMN:
            df.insert(0, 'anno', anno)

        all_dfs.append(df)
        print(f'  Colonne: {len(df.columns)}')

    except Exception as e:
        import traceback
        print(f'  ERRORE: {e}')
        traceback.print_exc()

print('\n' + '='*60)
if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)
    print(f'DataFrame totale: {len(df_all):,} righe × {len(df_all.columns)} colonne')
else:
    print('Nessun file elaborato con successo.')

## 5. Ispezione del DataFrame

In [ ]:
print(df_all.dtypes.to_string())

In [ ]:
df_all.head(3)

In [ ]:
# Riepilogo per anno
if ADD_YEAR_COLUMN:
    display(
        df_all.groupby('anno').size().rename('righe').reset_index()
    )

In [ ]:
# Anteprima campi con decodifica (es. mese, provincia)
cols_example = ['anno', 'mese', 'mese_label', 'provincia', 'provincia_label']
cols_present = [c for c in cols_example if c in df_all.columns]
if cols_present:
    display(df_all[cols_present].head(10))

---

## 7. Analisi: incidenti in ambito urbano con utenti deboli o scontri frontali

Filtri applicati:
- **`localizzazione_incidente`** in [0, 1, 2, 3]
- **`natura_incidente`** == `05` **oppure** almeno uno dei veicoli coinvolti (a/b/c) ha **`tipo_veicolo`** tra 14 e 17 (velocipede, motociclo, altri utenti deboli)

Aggregazione per **anno × provincia × comune**.

In [ ]:
# Il filtro è già stato applicato durante la lettura dei file.
# df_all contiene solo le righe con:
#   - localizzazione_incidente in [0, 1, 2, 3]
#   - natura_incidente == '05'  OPPURE  tipo_veicolo (a/b/c) in [14..17]
print(f'Righe totali nel dataset filtrato: {len(df_all):,}')
df_all.head(5)

In [ ]:
# ── Aggregazione per anno × provincia × comune ───────────────────────────────
df_analisi = (
    df_all
    .groupby(['anno', 'provincia', 'comune'], dropna=False)
    .agg(
        n_incidenti       = ('morti_entro_24_ore',   'count'),
        morti_24h         = ('morti_entro_24_ore',   'sum'),
        morti_30g         = ('morti_entro_30_giorni','sum'),
        feriti            = ('feriti',               'sum'),
    )
    .reset_index()
    .sort_values(['anno', 'provincia', 'comune'])
)

print(f'Righe nel risultato: {len(df_analisi):,}')
display(df_analisi.head(20))

### Tabella pivot: morti e feriti per città × anno

In [ ]:
# ── Dettaglio Bologna ─────────────────────────────────────────────────────────
cols_det = ['anno', 'provincia', 'comune', 'natura_incidente',
            'morti_entro_24_ore', 'morti_entro_30_giorni', 'feriti']
if 'comune_label' in df_all.columns:
    cols_det.insert(3, 'comune_label')
cols_det = [c for c in cols_det if c in df_all.columns]

mask_bo = df_all['comune_label'].astype(str).str.lower().str.contains('bologna') \
          if 'comune_label' in df_all.columns \
          else (df_all['provincia'].astype(str).str.strip() == '037')

det_bologna = df_all[mask_bo][cols_det].sort_values('anno')
print(f'Righe Bologna: {len(det_bologna):,}')
display(det_bologna)

In [ ]:
# ── Dettaglio Olbia ───────────────────────────────────────────────────────────
# Olbia ha cambiato provincia (Sassari → Olbia-Tempio): può apparire con
# più codici ISTAT. La ricerca per nome cattura tutte le varianti.
mask_ol = df_all['comune_label'].astype(str).str.lower().str.contains('olbia') \
          if 'comune_label' in df_all.columns \
          else pd.Series(False, index=df_all.index)

det_olbia = df_all[mask_ol][cols_det].sort_values('anno')
print(f'Righe Olbia: {len(det_olbia):,}')
print(f'Codici provincia×comune presenti: '
      f'{det_olbia[["provincia","comune"]].drop_duplicates().values.tolist()}')
display(det_olbia)

In [ ]:
# ── Pivot utenti deboli: pedoni (natura 05) o veicoli deboli (tipo 14-17) ─────
veic_cols_ud = [c for c in ['tipo_veicolo_a','tipo_veicoli__b_','tipo_veicolo__c_'] if c in df_all.columns]
nat_mask_ud  = df_all['natura_incidente'].astype(str).str.strip() == '05'
veic_mask_ud = (df_all[veic_cols_ud].apply(lambda s: s.astype(str).str.strip().isin({'14','15','16','17'})).any(axis=1)
                if veic_cols_ud else pd.Series(False, index=df_all.index))
df_ud = df_all[nat_mask_ud | veic_mask_ud].copy()
print(f'Righe utenti deboli: {len(df_ud):,} (su {len(df_all):,} incidenti urbani)')

tmp = df_ud.copy()
tmp['codice_istat'] = (
    tmp['provincia'].astype(str).str.strip().str.zfill(3) +
    tmp['comune'].astype(str).str.strip().str.zfill(3)
)
tmp['totale_morti'] = (
    pd.to_numeric(tmp['morti_entro_24_ore'],    errors='coerce').fillna(0) +
    pd.to_numeric(tmp['morti_entro_30_giorni'], errors='coerce').fillna(0)
).astype(int)
tmp['feriti_n'] = pd.to_numeric(tmp['feriti'], errors='coerce').fillna(0).astype(int)
tmp['n_inc'] = 1

agg = (
    tmp
    .groupby(['codice_istat', 'anno'], dropna=False)
    .agg(morti=('totale_morti','sum'), feriti=('feriti_n','sum'), incidenti=('n_inc','sum'))
    .reset_index()
)

pivot = agg.pivot_table(
    index='codice_istat', columns='anno',
    values=['morti','feriti','incidenti'], aggfunc='sum', fill_value=0,
)
pivot = pivot.swaplevel(axis=1).sort_index(axis=1)
pivot.columns = [f'{anno}_{met}' for anno, met in pivot.columns]
pivot = pivot.reset_index()

if 'comune_label' in tmp.columns:
    nomi = (
        tmp[['codice_istat','comune_label']].dropna(subset=['comune_label'])
        .query('comune_label != codice_istat')
        .assign(pulito=lambda x: x['comune_label'].str.match(r"^[A-Za-zÀ-ÿ '\-\.]+$"))
        .sort_values('pulito', ascending=False)
        .groupby('codice_istat')['comune_label'].first()
        .rename('nome_comune').reset_index()
    )
    pivot = pivot.merge(nomi, on='codice_istat', how='left')
    pivot['nome_comune'] = pivot['nome_comune'].fillna(pivot['codice_istat'])
    cols = ['codice_istat','nome_comune'] + [c for c in pivot.columns if c not in ('codice_istat','nome_comune')]
    pivot = pivot[cols]

val_cols = [c for c in pivot.columns if c not in ('codice_istat','nome_comune')]
rows = []
for nome, group in pivot.groupby('nome_comune', sort=False):
    if len(group) == 1:
        rows.append(group.iloc[0].to_dict())
        continue
    if (group[val_cols] > 0).sum(axis=0).max() <= 1:
        merged = group[val_cols].sum()
        row = {'codice_istat': '/'.join(group['codice_istat'].tolist()), 'nome_comune': nome}
        row.update(merged.to_dict())
        rows.append(row)
    else:
        for _, r in group.iterrows():
            rows.append(r.to_dict())

pivot = pd.DataFrame(rows, columns=['codice_istat','nome_comune'] + val_cols)
n_fusi = pivot['codice_istat'].str.contains('/').sum()
print(f'Comuni presenti: {len(pivot):,}  |  di cui fusi: {n_fusi}')
print(f'Anni: {sorted(agg["anno"].unique())}')
display(pivot)

In [ ]:
import os
output_dir = '/content/drive/MyDrive/TesiMagistrale/FontiUsate'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'utenti_deboli.csv')
pivot.to_csv(output_path, index=False, encoding='utf-8-sig', sep=';')
print(f'Salvato: {output_path}')

In [ ]:
# ── Pivot pedoni: solo investimenti di pedone (natura_incidente == '05') ──────
df_pedoni = df_all[df_all['natura_incidente'].astype(str).str.strip() == '05'].copy()
print(f'Righe investimento pedone: {len(df_pedoni):,}')

tmp_p = df_pedoni.copy()
tmp_p['codice_istat'] = (
    tmp_p['provincia'].astype(str).str.strip().str.zfill(3) +
    tmp_p['comune'].astype(str).str.strip().str.zfill(3)
)
tmp_p['totale_morti'] = (
    pd.to_numeric(tmp_p['morti_entro_24_ore'],    errors='coerce').fillna(0) +
    pd.to_numeric(tmp_p['morti_entro_30_giorni'], errors='coerce').fillna(0)
).astype(int)
tmp_p['feriti_n'] = pd.to_numeric(tmp_p['feriti'], errors='coerce').fillna(0).astype(int)
tmp_p['n_inc'] = 1

agg_p = (
    tmp_p.groupby(['codice_istat', 'anno'], dropna=False)
    .agg(morti=('totale_morti','sum'), feriti=('feriti_n','sum'), incidenti=('n_inc','sum'))
    .reset_index()
)

pivot_p = agg_p.pivot_table(
    index='codice_istat', columns='anno',
    values=['morti','feriti','incidenti'], aggfunc='sum', fill_value=0,
)
pivot_p = pivot_p.swaplevel(axis=1).sort_index(axis=1)
pivot_p.columns = [f'{anno}_{met}' for anno, met in pivot_p.columns]
pivot_p = pivot_p.reset_index()

# Nome comune
if 'comune_label' in tmp_p.columns:
    nomi_p = (
        tmp_p[['codice_istat','comune_label']].dropna(subset=['comune_label'])
        .query('comune_label != codice_istat')
        .assign(pulito=lambda x: x['comune_label'].str.match(r"^[A-Za-zÀ-ÿ '\-\.]+$"))
        .sort_values('pulito', ascending=False)
        .groupby('codice_istat')['comune_label'].first()
        .rename('nome_comune').reset_index()
    )
    pivot_p = pivot_p.merge(nomi_p, on='codice_istat', how='left')
    pivot_p['nome_comune'] = pivot_p['nome_comune'].fillna(pivot_p['codice_istat'])
    cols_p = ['codice_istat','nome_comune'] + [c for c in pivot_p.columns if c not in ('codice_istat','nome_comune')]
    pivot_p = pivot_p[cols_p]

# Fusione comuni con codice cambiato
val_cols_p = [c for c in pivot_p.columns if c not in ('codice_istat','nome_comune')]
rows_p = []
for nome, group in pivot_p.groupby('nome_comune', sort=False):
    if len(group) == 1:
        rows_p.append(group.iloc[0].to_dict())
        continue
    if (group[val_cols_p] > 0).sum(axis=0).max() <= 1:
        merged = group[val_cols_p].sum()
        row = {'codice_istat': '/'.join(group['codice_istat'].tolist()), 'nome_comune': nome}
        row.update(merged.to_dict())
        rows_p.append(row)
    else:
        for _, r in group.iterrows():
            rows_p.append(r.to_dict())

pivot_p = pd.DataFrame(rows_p, columns=['codice_istat','nome_comune'] + val_cols_p)
n_fusi_p = pivot_p['codice_istat'].str.contains('/').sum()
print(f'Comuni presenti: {len(pivot_p):,}  |  di cui fusi: {n_fusi_p}')
print(f'Anni: {sorted(agg_p["anno"].unique())}')
display(pivot_p)

In [ ]:
output_path_p = os.path.join(output_dir, 'pedoni.csv')
pivot_p.to_csv(output_path_p, index=False, encoding='utf-8-sig', sep=';')
print(f'Salvato: {output_path_p}')

In [ ]:
# ── Pivot incidenti urbani: tutti gli incidenti con localizzazione [0-3] ──────
# df_all contiene già solo le righe con localizzazione_incidente in [0,1,2,3]
# nessun filtro aggiuntivo su natura o tipo_veicolo
print(f'Righe incidenti urbani (df_all): {len(df_all):,}')

tmp_u = df_all.copy()
tmp_u['codice_istat'] = (
    tmp_u['provincia'].astype(str).str.strip().str.zfill(3) +
    tmp_u['comune'].astype(str).str.strip().str.zfill(3)
)
tmp_u['totale_morti'] = (
    pd.to_numeric(tmp_u['morti_entro_24_ore'],    errors='coerce').fillna(0) +
    pd.to_numeric(tmp_u['morti_entro_30_giorni'], errors='coerce').fillna(0)
).astype(int)
tmp_u['feriti_n'] = pd.to_numeric(tmp_u['feriti'], errors='coerce').fillna(0).astype(int)
tmp_u['n_inc'] = 1

agg_u = (
    tmp_u.groupby(['codice_istat', 'anno'], dropna=False)
    .agg(morti=('totale_morti', 'sum'), feriti=('feriti_n', 'sum'), incidenti=('n_inc', 'sum'))
    .reset_index()
)

pivot_u = agg_u.pivot_table(
    index='codice_istat', columns='anno',
    values=['morti', 'feriti', 'incidenti'], aggfunc='sum', fill_value=0,
)
pivot_u = pivot_u.swaplevel(axis=1).sort_index(axis=1)
pivot_u.columns = [f'{anno}_{met}' for anno, met in pivot_u.columns]
pivot_u = pivot_u.reset_index()

# Nome comune (stessa logica di utenti_deboli)
if 'comune_label' in tmp_u.columns:
    nomi_u = (
        tmp_u[['codice_istat', 'comune_label']].dropna(subset=['comune_label'])
        .query('comune_label != codice_istat')
        .assign(pulito=lambda x: x['comune_label'].str.match(r"^[A-Za-zÀ-ÿ '\-\.]+$"))
        .sort_values('pulito', ascending=False)
        .groupby('codice_istat')['comune_label'].first()
        .rename('nome_comune').reset_index()
    )
    pivot_u = pivot_u.merge(nomi_u, on='codice_istat', how='left')
    pivot_u['nome_comune'] = pivot_u['nome_comune'].fillna(pivot_u['codice_istat'])
    cols_u = ['codice_istat', 'nome_comune'] + [c for c in pivot_u.columns if c not in ('codice_istat', 'nome_comune')]
    pivot_u = pivot_u[cols_u]

# Fusione comuni con codice ISTAT cambiato nel tempo
val_cols_u = [c for c in pivot_u.columns if c not in ('codice_istat', 'nome_comune')]
rows_u = []
for nome, group in pivot_u.groupby('nome_comune', sort=False):
    if len(group) == 1:
        rows_u.append(group.iloc[0].to_dict())
        continue
    if (group[val_cols_u] > 0).sum(axis=0).max() <= 1:
        merged = group[val_cols_u].sum()
        row = {'codice_istat': '/'.join(group['codice_istat'].tolist()), 'nome_comune': nome}
        row.update(merged.to_dict())
        rows_u.append(row)
    else:
        for _, r in group.iterrows():
            rows_u.append(r.to_dict())

pivot_u = pd.DataFrame(rows_u, columns=['codice_istat', 'nome_comune'] + val_cols_u)
n_fusi_u = len(pivot_u[pivot_u['codice_istat'].str.contains('/')])
print(f'Comuni presenti: {len(pivot_u):,}  |  Comuni fusi (codice cambiato): {n_fusi_u}  |  Anni: {sorted(agg_u["anno"].unique())}')
display(pivot_u)

In [ ]:
output_path_u = os.path.join(output_dir, 'incidenti_urbani.csv')
pivot_u.to_csv(output_path_u, index=False, encoding='utf-8-sig', sep=';')
print(f'Salvato: {output_path_u}')

In [ ]:
# ── Diagnostica: colonne disponibili per anno 2023/2024 ──────────────────────
# Eseguire questa cella PRIMA di Bologna2023/2024 per verificare i nomi dei campi.
# Se giorno_settimana e ora risultano tutti NaN, il tracciato di quell'anno
# usa nomi diversi: cerca la colonna con "giorno" o "ora" nel nome.

print("Colonne in df_all:", list(df_all.columns))
print()
for yr in [2023, 2024]:
    sub = df_all[df_all['anno'] == yr]
    print(f"Anno {yr}: {len(sub):,} righe totali")
    if len(sub) == 0:
        continue
    # Mostra colonne con almeno un valore non nullo
    for col in sub.columns:
        n_non_null = sub[col].notna().sum()
        pct = n_non_null / len(sub) * 100
        flag = '' if n_non_null > 0 else ' ← VUOTA'
        if 'giorno' in col.lower() or 'ora' in col.lower() or 'mese' in col.lower() or n_non_null == 0:
            print(f"  {col}: {n_non_null:,}/{len(sub):,} ({pct:.0f}%){flag}")
    print()


In [ ]:
# ── Bologna 2023 e 2024: incidenti urbani per giorno settimana, ora e trimestre ──
# Filtro: comune di Bologna = provincia 037, comune 006
mask_bo = (
    (df_all['provincia'].astype(str).str.strip() == '037') &
    (df_all['comune'].astype(str).str.strip() == '006')
)
df_bologna = df_all[mask_bo].copy()
print(f'Righe Bologna città (tutti gli anni): {len(df_bologna):,}')

# Gestione nomi alternativi del campo giorno della settimana e ora
# (ISTAT può variare il "Nome Campo" nel tracciato tra anni diversi)
def _find_col(df, candidates):
    """Restituisce il primo nome di colonna trovato tra i candidati."""
    for c in candidates:
        if c in df.columns and df[c].notna().any():
            return c
    # Fallback: cerca per substring
    for c in df.columns:
        for cand in candidates:
            if cand.lower() in c.lower() and df[c].notna().any():
                return c
    return None

col_giorno = _find_col(df_bologna, ['giorno_settimana', 'giorno'])
col_ora    = _find_col(df_bologna, ['ora', 'ora_incidente'])

print(f'Colonna giorno settimana rilevata: {col_giorno}')
print(f'Colonna ora rilevata             : {col_ora}')

# Rinomina verso i nomi attesi se necessario
rename_map = {}
if col_giorno and col_giorno != 'giorno_settimana':
    rename_map[col_giorno] = 'giorno_settimana'
if col_ora and col_ora != 'ora':
    rename_map[col_ora] = 'ora'
if rename_map:
    df_bologna = df_bologna.rename(columns=rename_map)

# Colonne di output desiderate
_output_cols = [
    'provincia', 'comune', 'giorno_settimana', 'localizzazione_incidente',
    'morti_entro_24_ore', 'morti_entro_30_giorni', 'feriti', 'ora', 'trimestre',
]

for target_anno in [2023, 2024]:
    df_year = df_bologna[df_bologna['anno'] == target_anno].copy()

    # Calcola trimestre dal mese (1=gen-mar, 2=apr-giu, 3=lug-set, 4=ott-dic)
    df_year['trimestre'] = (
        (pd.to_numeric(df_year['mese'], errors='coerce') - 1) // 3 + 1
    ).astype('Int64')

    cols_present = [c for c in _output_cols if c in df_year.columns]
    out_path = os.path.join(output_dir, f'Bologna{target_anno}.csv')
    df_year[cols_present].to_csv(out_path, index=False, encoding='utf-8-sig', sep=';')
    print(f'Salvato: {out_path}  ({len(df_year):,} righe)')


### Verifica: `morti_entro_30_giorni` include o esclude `morti_entro_24_ore`?

- Se esiste almeno un record con `morti_entro_24_ore > morti_entro_30_giorni` → i due campi sono **indipendenti** (30gg = solo decessi tra 24h e 30gg)
- Se non esiste mai tale condizione → `morti_entro_30_giorni` è probabilmente **cumulativo** (include anche i morti nelle prime 24h)

In [ ]:
m24 = pd.to_numeric(df_all['morti_entro_24_ore'],   errors='coerce')
m30 = pd.to_numeric(df_all['morti_entro_30_giorni'], errors='coerce')

# Righe con almeno un morto (esclude i casi banali 0 == 0)
has_deaths = (m24 > 0) | (m30 > 0)

n_24_gt_30 = ((m24 > m30) & has_deaths).sum()
n_24_eq_30 = ((m24 == m30) & has_deaths).sum()
n_30_gt_24 = ((m30 > m24) & has_deaths).sum()
n_both_zero = (~has_deaths).sum()

print(f'Record con almeno un morto          : {has_deaths.sum():>10,}')
print(f'  morti_24h  > morti_30g            : {n_24_gt_30:>10,}')
print(f'  morti_24h == morti_30g  (entrambi>0): {n_24_eq_30:>10,}')
print(f'  morti_30g  > morti_24h            : {n_30_gt_24:>10,}')
print(f'Record con entrambi a zero          : {n_both_zero:>10,}')
print()

if n_24_gt_30 > 0:
    print('✔ Esiste almeno un record con morti_24h > morti_30g.')
    print('  → I due campi sono INDIPENDENTI:')
    print('    morti_entro_24_ore  = decessi nelle prime 24h')
    print('    morti_entro_30_giorni = decessi tra 24h e 30gg (escluse le 24h)')
    print()
    print('  Totale decessi = morti_entro_24_ore + morti_entro_30_giorni')
else:
    print('✘ Non esiste alcun record con morti_24h > morti_30g.')
    print('  → I due campi sono probabilmente CUMULATIVI:')
    print('    morti_entro_24_ore  = decessi nelle prime 24h')
    print('    morti_entro_30_giorni = decessi totali entro 30gg (include le 24h)')
    print()
    print('  Totale decessi = morti_entro_30_giorni (già comprensivo delle 24h)')

# Mostra qualche esempio con morti in entrambe le colonne
sample = df_all[has_deaths][['anno', 'provincia', 'comune',
                               'morti_entro_24_ore', 'morti_entro_30_giorni']].head(10)
print()
display(sample)

---

## Bologna 2023 – Incidenti urbani

Lettura diretta del file 2023, indipendente dal pipeline generale.

Filtri: `provincia == '037'`, `comune == '006'`, `localizzazione_incidente` in [0, 1, 2, 3].

In [ ]:
# ── Lettura microdati 2023 e filtro Bologna ───────────────────────────────────
_DATA_DIR = '/content/drive/MyDrive/TesiMagistrale/dati'
_filepath  = os.path.join(_DATA_DIR, 'INCSTRAD_2023_IT', 'MICRODATI', 'INCSTRAD_Microdati_2023.txt')
_tracciato = os.path.join(_DATA_DIR, 'INCSTRAD_2023_IT', 'METADATI', 'INCSTRAD_Tracciato_2023.html')

print(f'File dati : {_filepath}')
print(f'Tracciato : {_tracciato}')

# 1. Parse tracciato → nomi colonne
_fields_2023, _ = parse_tracciato(_tracciato)
_col_names = [f['nome'] for f in _fields_2023]
print(f'\nColonne rilevate dal tracciato: {_col_names}')

# 2. Leggi file TAB-separated (tutte le colonne come stringhe)
_df_raw = pd.read_csv(
    _filepath, sep='\t', names=_col_names,
    header=None, encoding='latin-1', dtype=str, on_bad_lines='warn',
)
print(f'\nRighe totali nel file 2023: {len(_df_raw):,}')

# 3. Filtro Bologna città + localizzazione urbana [0-3]
_mask_bo  = (
    _df_raw['provincia'].str.strip() == '037'
) & (
    _df_raw['comune'].str.strip() == '006'
)
_mask_urb = _df_raw['localizzazione_incidente'].str.strip().isin(['0','1','2','3'])

df_bo23 = _df_raw[_mask_bo & _mask_urb].copy()
print(f'Righe Bologna 2023 (urbano): {len(df_bo23):,}')


In [ ]:
# ── Selezione colonne e calcolo trimestre ─────────────────────────────────────
# trimestre ricavato da mese: 1=gen-mar, 2=apr-giu, 3=lug-set, 4=ott-dic
df_bo23['trimestre'] = (
    (pd.to_numeric(df_bo23['mese'], errors='coerce') - 1) // 3 + 1
).astype('Int64')

_COLS_OUT = [
    'provincia', 'comune', 'giorno_settimana', 'localizzazione_incidente',
    'morti_entro_24_ore', 'morti_entro_30_giorni', 'feriti', 'ora', 'trimestre',
]

_cols_missing = [c for c in _COLS_OUT if c not in df_bo23.columns]
if _cols_missing:
    print(f'⚠ Colonne assenti nel tracciato 2023: {_cols_missing}')

_cols_ok = [c for c in _COLS_OUT if c in df_bo23.columns]
df_bo23 = df_bo23[_cols_ok]
print(f'Colonne selezionate: {_cols_ok}')
display(df_bo23.head())


In [ ]:
# ── Verifica: nessun campo vuoto ─────────────────────────────────────────────
print(f'Righe totali: {len(df_bo23):,}\n')
_tutti_ok = True
for _col in df_bo23.columns:
    _series = df_bo23[_col]
    # conta NaN e stringhe vuote/nan
    _n_bad = (
        _series.isna() |
        _series.astype(str).str.strip().isin(['', 'nan', '<NA>'])
    ).sum()
    _flag = '✔' if _n_bad == 0 else '⚠'
    if _n_bad > 0:
        _tutti_ok = False
    print(f'  {_flag} {_col}: {_n_bad} vuoti/nulli')

print()
if _tutti_ok:
    print('✔ Tutti i campi sono completi.')
else:
    print('⚠ Alcuni campi hanno valori mancanti — verificare il tracciato 2023.')


In [ ]:
# ── Salva Bologna2023.csv ─────────────────────────────────────────────────────
_output_dir = '/content/drive/MyDrive/TesiMagistrale/FontiUsate'
os.makedirs(_output_dir, exist_ok=True)
_out_path = os.path.join(_output_dir, 'Bologna2023.csv')
df_bo23.to_csv(_out_path, index=False, encoding='utf-8-sig', sep=';')
print(f'Salvato: {_out_path}')
print(f'Righe: {len(df_bo23):,}  |  Colonne: {list(df_bo23.columns)}')
